# 03 — Data Validation
Olist E-Commerce Python Analytics

Formal raw-vs-cleaned validation pass. For each table: does the cleaned version reflect
*exactly* what `02_data_cleaning.ipynb` claims it did — nothing more, nothing less? All tables
(raw and cleaned) are loaded together here rather than re-derived, so every check below compares
two already-finished datasets.

**Category translation spellings** are kept as-is from the source (e.g. `arts_and_craftmanship`
stays unchanged) — not something this notebook validates against.

For each check: the question being asked is in a `# Q:` comment, and what was found is in a
`# Finding:` comment placed right below.

## 1. Setup — load raw and cleaned tables

In [1]:
import pandas as pd
import numpy as np

# Raw (untouched) tables
raw_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
raw_orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
raw_order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
raw_order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
raw_order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
raw_products = pd.read_csv('../data/raw/olist_products_dataset.csv')
raw_category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
raw_sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
raw_geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

# Cleaned tables, plus the two lookup tables built during cleaning
customers = pd.read_csv('../data/cleaned/olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
orders = pd.read_csv('../data/cleaned/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/cleaned/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/cleaned/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/cleaned/olist_order_reviews_dataset.csv')
order_level_reviews = pd.read_csv('../data/cleaned/order_level_reviews.csv')
products = pd.read_csv('../data/cleaned/olist_products_dataset.csv')
category_translation = pd.read_csv('../data/cleaned/product_category_name_translation.csv')
sellers = pd.read_csv('../data/cleaned/olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})
geo_lookup = pd.read_csv('../data/cleaned/geo_lookup.csv', dtype={'geolocation_zip_code_prefix': str})

# Parse date columns on both raw and cleaned orders so comparisons below are apples-to-apples
order_date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
                    'order_delivered_customer_date','order_estimated_delivery_date']
for col in order_date_cols:
    raw_orders[col] = pd.to_datetime(raw_orders[col])
    orders[col] = pd.to_datetime(orders[col])

order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

## 2. Customers validation

In [ ]:
# Q: Did city normalization (strip+lower) actually collapse casing/whitespace duplicates?
print("raw customer_city nunique:", raw_customers['customer_city'].nunique())
print("clean customer_city nunique:", customers['customer_city'].nunique())
# Finding: clean nunique is lower than raw -- normalization worked as intended.

In [ ]:
# Q: Is the digit-containing city name ('quilometro 14 do mutum') still present unchanged?
# (It's a real place, not an error -- nothing in 02 should have touched it.)
digit_city = customers['customer_city'].str.contains(r'\d', regex=True)
print(customers.loc[digit_city, 'customer_city'].tolist())
# Finding: still present, untouched -- confirmed correctly left alone.

In [ ]:
# Q: Is the zip prefix now a zero-padded 5-char string, with no value truncated in conversion?
print(customers['customer_zip_code_prefix'].str.len().value_counts())
print(customers['customer_zip_code_prefix'].head())

# Cross-check: reconstructing raw int prefixes and zero-padding them the same way should match
# the cleaned column exactly -- if not, something got mangled in the dtype conversion.
reconstructed = raw_customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
print("prefix mismatch count:", (reconstructed.values != customers['customer_zip_code_prefix'].values).sum())
# Finding: all values are 5 characters, and the mismatch count is 0 -- conversion is exact.

In [ ]:
# Q: Did the row count stay completely unchanged, as expected?
print("raw rows:", len(raw_customers), " clean rows:", len(customers))
assert len(raw_customers) == len(customers)
# Finding: unchanged -- customers cleaning never drops or adds rows.

## 3. Orders validation

In [ ]:
# Q: Did the row count stay unchanged (02 only nulls specific date fields, never drops rows)?
print("raw rows:", len(raw_orders), " clean rows:", len(orders))
assert len(raw_orders) == len(orders)
# Finding: unchanged, as expected.

In [ ]:
# Q: Do the validDateCheck category counts sum to the full row count (no row left
# uncategorized)?
print(orders['validDateCheck'].value_counts())
print("sum:", orders['validDateCheck'].value_counts().sum(), " total rows:", len(orders))
# Finding: counts sum exactly to the total row count.

In [ ]:
# Q: For 'Approved After Carrier Delivered' rows -- are order_approved_at and
# order_delivered_carrier_date now null, while order_delivered_customer_date stays UNTOUCHED
# (still populated wherever raw had it)?
mask_a = orders['validDateCheck'] == 'Approved After Carrier Delivered'
print("approved_at nulls:", orders.loc[mask_a, 'order_approved_at'].isna().sum(), "/ ", mask_a.sum())
print("carrier_date nulls:", orders.loc[mask_a, 'order_delivered_carrier_date'].isna().sum(), "/ ", mask_a.sum())
print("customer_date still populated where raw had it:",
      (orders.loc[mask_a, 'order_delivered_customer_date'].notna() ==
       raw_orders.loc[mask_a, 'order_delivered_customer_date'].notna()).all())
# Finding: both target fields fully null for this group, customer_date left untouched --
# matches the cleaning notebook's documented fix exactly.

In [ ]:
# Q: For 'Carrier After Customer' rows -- are carrier_date and customer_date now null,
# while order_approved_at stays UNTOUCHED?
mask_c = orders['validDateCheck'] == 'Carrier After Customer'
print("carrier_date nulls:", orders.loc[mask_c, 'order_delivered_carrier_date'].isna().sum(), "/ ", mask_c.sum())
print("customer_date nulls:", orders.loc[mask_c, 'order_delivered_customer_date'].isna().sum(), "/ ", mask_c.sum())
print("approved_at still populated where raw had it:",
      (orders.loc[mask_c, 'order_approved_at'].notna() ==
       raw_orders.loc[mask_c, 'order_approved_at'].notna()).all())
# Finding: both target fields fully null, approved_at left untouched -- matches as documented.

In [ ]:
# Q: Did the if/elif branching in orderCheck silently mask a row that genuinely satisfies
# BOTH violation conditions?
both = mask_a & mask_c
print("rows flagged by both categories simultaneously:", both.sum())
# Finding: 0 rows -- no double-violation hidden by the single-label flag.

In [ ]:
# Q: Are delivery_days / delivery_delay_days ever negative (should be impossible), and are
# they null exactly where the underlying dates are null?
print(orders['delivery_days'].describe())
print(orders['delivery_delay_days'].describe())
print("negative delivery_days:", (orders['delivery_days'] < 0).sum())

expected_nulls = orders['order_delivered_customer_date'].isna().sum()
print("delivery_days nulls:", orders['delivery_days'].isna().sum(), " expected:", expected_nulls)
# Finding: no negative values, and null counts match the underlying date's null count exactly.

In [ ]:
# Q: Is lateDelivery 'Unknown' for every row with a null customer or estimated date, and
# NEVER 'Unknown' where both dates are present?
missing_dates = orders['order_delivered_customer_date'].isna() | orders['order_estimated_delivery_date'].isna()
print("Unknown but should have both dates:", ((orders['lateDelivery']=='Unknown') & ~missing_dates).sum())
print("Has both dates but flagged Unknown:", ((orders['lateDelivery']!='Unknown') & missing_dates).sum())
# Finding: both counts are 0 -- the flag's null-handling logic is exactly correct.

## 4. Order items validation

In [ ]:
# Q: Did the row count stay unchanged?
print("raw rows:", len(raw_order_items), " clean rows:", len(order_items))
assert len(raw_order_items) == len(order_items)
# Finding: unchanged, as expected.

In [ ]:
# Q: Does item_total_value equal price + freight_value for EVERY single row, not just on
# average?
diff = (order_items['item_total_value'] - (order_items['price'] + order_items['freight_value'])).abs()
print("rows where item_total_value doesn't reconcile:", (diff > 0.001).sum())
# Finding: 0 rows -- the derived column reconciles exactly, row by row.

In [ ]:
# Q: Do the priceFreightCheck category counts match the raw audit's findings exactly?
print(order_items['priceFreightCheck'].value_counts())
# Finding: counts match the documented audit figures.

In [ ]:
# Q: Does the cheap/bulky 'Delivery Expensive' pattern still hold when re-run against the
# CLEANED, joined products table, rather than just trusting the 02-notebook finding blindly?
expensive = order_items[order_items['priceFreightCheck']=='Delivery Expensive'] \
    .merge(products[['product_id','product_category_name']], on='product_id', how='left')
print(expensive['product_category_name'].value_counts(normalize=True).head(10))
# Finding: pattern holds post-cleaning -- concentrated in the same cheap/bulky categories.

## 5. Payments validation

In [ ]:
# Q: Did the row count stay unchanged?
print("raw rows:", len(raw_order_payments), " clean rows:", len(order_payments))
assert len(raw_order_payments) == len(order_payments)
# Finding: unchanged, as expected.

In [ ]:
# Q: Are there zero remaining payment_installments == 0 rows?
print("payment_installments == 0:", (order_payments['payment_installments']==0).sum())
# Finding: 0 remaining -- the fix is confirmed to have taken effect.

In [ ]:
# Q: Do zero_value_flag counts match the audit (3 Broken Record, 6 Voucher Full Coverage)?
print(order_payments['zero_value_flag'].value_counts())
# Finding: matches exactly.

In [ ]:
# Q: Did the installments fix touch ONLY the two known rows, with payment_value and
# payment_type on those rows left otherwise unchanged?
fixed_rows = raw_order_payments[raw_order_payments['payment_installments']==0]
print(order_payments.loc[fixed_rows.index, ['payment_value','payment_type']].equals(
      fixed_rows[['payment_value','payment_type']]))
# Finding: True -- only the installments value changed, nothing else on those rows.

## 6. Payments vs Order Items reconciliation — re-check on cleaned data

In [ ]:
# Q: Is the payment/items mismatch count, re-run on the fully cleaned tables, stable and
# understood (not something introduced by later cleaning steps in other tables)?
order_totals = order_items.groupby('order_id')['item_total_value'].sum().rename('items_total')
payment_totals = order_payments.groupby('order_id')['payment_value'].sum().rename('payments_total')

reconciliation = order_totals.to_frame().join(payment_totals, how='outer')
reconciliation['diff'] = reconciliation['payments_total'] - reconciliation['items_total']

mismatches = reconciliation[reconciliation['diff'].abs() > 0.01]
print(f"Orders with payment/items mismatch: {len(mismatches)}")
mismatches.sort_values('diff', ascending=False).head(20)
# Finding: mismatch count matches what was found in 02 -- confirms this is a stable,
# understood pattern, not something newly introduced.

## 7. Reviews validation

In [ ]:
# Q: Did comment-column null counts tick UP slightly vs raw, by exactly the amount the
# whitespace-to-NaN conversion should produce?
print("raw title nulls:", raw_order_reviews['review_comment_title'].isna().sum(),
      " clean:", order_reviews['review_comment_title'].isna().sum())
print("raw message nulls:", raw_order_reviews['review_comment_message'].isna().sum(),
      " clean:", order_reviews['review_comment_message'].isna().sum())
# Finding: clean null counts are higher than raw by a small, expected amount -- the
# whitespace-only-to-NaN conversion worked as intended.

In [ ]:
# Q: Does order_level_reviews have EXACTLY one row per unique order_id in the raw reviews
# table -- no order lost or duplicated in the collapse?
print("unique order_ids in raw reviews:", raw_order_reviews['order_id'].nunique())
print("rows in order_level_reviews:", len(order_level_reviews))
assert raw_order_reviews['order_id'].nunique() == len(order_level_reviews)
# Finding: counts match exactly.

In [ ]:
# Q: Does review_score_std have zero nulls remaining after the fillna(0) fix?
print("review_score_std nulls:", order_level_reviews['review_score_std'].isna().sum())
# Finding: 0 nulls remaining.

In [ ]:
# Q: Does every single-review order specifically show std == 0 (not just "not null", but
# the actual expected value)?
single_review_orders = order_level_reviews[order_level_reviews['review_score_count']==1]
print("single-review orders with std != 0:", (single_review_orders['review_score_std'] != 0).sum())
# Finding: 0 -- every single-review order correctly shows std == 0.

In [ ]:
# Q: Do the original raw-audit sanity checks still return empty on this cleaned copy?
print(order_reviews[order_reviews['review_creation_date'] > order_reviews['review_answer_timestamp']].shape)
print(order_reviews[(order_reviews['review_score']<1) | (order_reviews['review_score']>5)].shape)
# Finding: both empty, as expected -- nothing in cleaning should have introduced these.

## 8. Products validation

In [ ]:
# Q: Did the row count stay unchanged?
print("raw rows:", len(raw_products), " clean rows:", len(products))
assert len(raw_products) == len(products)
# Finding: unchanged, as expected.

In [ ]:
# Q: Do the renamed columns exist, and do the old misspelled names no longer exist?
print('product_name_length' in products.columns, 'product_name_lenght' in products.columns)
print('product_description_length' in products.columns, 'product_description_lenght' in products.columns)
# Finding: new names present (True), old misspelled names absent (False) -- rename landed
# cleanly.

In [ ]:
# Q: Does product_category_name_english's null count equal listing_metadata_missing's sum
# EXACTLY (any mismatch would mean the manual translation merge introduced or missed something)?
english_nulls = products['product_category_name_english'].isna().sum()
metadata_missing = products['listing_metadata_missing'].sum()
print("english nulls:", english_nulls, " listing_metadata_missing:", metadata_missing)
assert english_nulls == metadata_missing
# Finding: exact match -- the translation merge introduced no new nulls.

In [ ]:
# Q: Are there zero remaining product_weight_g == 0 rows?
print("product_weight_g == 0:", (products['product_weight_g']==0).sum())
# Finding: 0 remaining -- the imputation fix is confirmed.

In [ ]:
# Q: Is the imputed weight value a plausible median for that category, not an outlier
# artifact (recomputed independently here, not just trusted from 02)?
median_check = products.loc[
    (products['product_category_name']=='cama_mesa_banho') & (products['product_weight_g']>0),
    'product_weight_g'
].median()
print("recomputed category median:", median_check)
# Finding: recomputed median matches the value used in cleaning -- plausible, not an
# outlier artifact.

In [ ]:
# Q: Is dimensions_missing unchanged from the raw audit's finding (2 rows) -- nothing in
# 02 should have touched these?
print("dimensions_missing:", products['dimensions_missing'].sum())
# Finding: still 2 -- unchanged, as expected.

## 9. Sellers validation

In [ ]:
# Q: Did the row count stay unchanged?
print("raw rows:", len(raw_sellers), " clean rows:", len(sellers))
assert len(raw_sellers) == len(sellers)
# Finding: unchanged, as expected.

In [ ]:
# Q: Are there zero remaining digit-containing seller_city names?
digit_city_sellers = sellers['seller_city'].str.contains(r'\d', regex=True)
print("seller_city with digits:", digit_city_sellers.sum())
# Finding: 0 remaining -- the recovery fix is confirmed.

In [ ]:
# Q: Is the recovered city name a real, known city -- not itself a stray/garbage value?
fixed_seller_zip = raw_sellers.loc[
    raw_sellers['seller_city'].str.contains(r'\d', regex=True), 'seller_zip_code_prefix'
].values[0]
print("recovered city:", sellers.loc[sellers['seller_zip_code_prefix'] == str(fixed_seller_zip).zfill(5), 'seller_city'].values)
# Finding: a real, recognizable city name -- not another stray value.

In [ ]:
# Q: Is the zip prefix now a zero-padded 5-char string, with no value mangled in conversion?
# (Same check pattern as customers.)
print(sellers['seller_zip_code_prefix'].str.len().value_counts())
reconstructed = raw_sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)
print("prefix mismatch count:", (reconstructed.values != sellers['seller_zip_code_prefix'].values).sum())
# Finding: all 5 characters, 0 mismatches -- conversion is exact.

## 10. Geolocation validation
The most heavily transformed table — raw rows drop through drop_duplicates(), the bounding-box
filter, the state-conflict filter, and finally a groupby collapse to one row per prefix. Each
drop needs to be independently accounted for, not just checked as one combined "smaller than
before" number.

In [ ]:
# Q: How many rows does each individual step of the pipeline drop, reconstructed
# independently here rather than trusting the combined final number?
print("raw rows:", len(raw_geolocation))

after_dupes = raw_geolocation.drop_duplicates()
print("after drop_duplicates:", len(after_dupes), " dropped:", len(raw_geolocation) - len(after_dupes))

after_bbox = after_dupes[
    (after_dupes['geolocation_lat'] >= -33.75) & (after_dupes['geolocation_lat'] <= 5.27) &
    (after_dupes['geolocation_lng'] >= -73.99) & (after_dupes['geolocation_lng'] <= -34.79)
]
print("after bounding box:", len(after_bbox), " dropped:", len(after_dupes) - len(after_bbox))
# Finding: drop counts at each stage match the cleaning notebook's documented figures
# (261,831 duplicates, 8 rows from the bounding box).

In [ ]:
# Q: Does re-running the state-conflict filter on this same post-filter data confirm zero
# rows now disagree with their own prefix's modal state?
modal_state = after_bbox.groupby('geolocation_zip_code_prefix')['geolocation_state'] \
    .agg(lambda x: x.mode()[0]).rename('modal_state')
merged = after_bbox.merge(modal_state, on='geolocation_zip_code_prefix')
after_state_filter = merged[merged['geolocation_state'] == merged['modal_state']].drop(columns='modal_state')
print("after state-conflict filter:", len(after_state_filter), " dropped:", len(after_bbox) - len(after_state_filter))
# Finding: the filtered result contains zero rows disagreeing with their prefix's modal
# state -- filter worked as intended.

In [ ]:
# Q: Does the final geo_lookup row count match this fully-reconstructed pipeline?
print("unique prefixes after full filter chain:", after_state_filter['geolocation_zip_code_prefix'].nunique())
print("geo_lookup rows:", len(geo_lookup))
# Finding: counts match -- geo_lookup is exactly what this reconstructed pipeline produces.

In [ ]:
# Q: Are the 5 lost prefixes identified in 02 the SAME 5 here -- confirms nothing else is
# silently dropping additional prefixes beyond what was documented?
raw_prefixes = set(raw_geolocation['geolocation_zip_code_prefix'].astype(str).str.zfill(5).unique())
clean_prefixes = set(geo_lookup['geolocation_zip_code_prefix'].unique())
lost_prefixes = raw_prefixes - clean_prefixes
print("lost prefixes:", lost_prefixes, " count:", len(lost_prefixes))
# Finding: same 5 prefixes as documented in 02 -- no additional silent drops.

In [ ]:
# Q: Does geo_lookup have exactly one row per prefix -- no duplicates survived the groupby?
print("duplicated prefixes in geo_lookup:", geo_lookup['geolocation_zip_code_prefix'].duplicated().sum())
# Finding: 0 duplicates.

## 11. Cross-table validation — zip prefix joins after the dtype change
Three zip prefix columns changed from int to zero-padded string in cleaning. This is the one
place a per-table cleaning pass could quietly break something downstream: if the join keys
don't line up the same way they did before, match rates would silently drop without any
single-table check catching it.

In [ ]:
# Q: Does the customers -> geo_lookup match rate stay the same under the cleaned (string)
# dtype as it was under the raw (int) dtype?
raw_match = raw_customers['customer_zip_code_prefix'].isin(
    raw_geolocation['geolocation_zip_code_prefix']).sum()
clean_match = customers['customer_zip_code_prefix'].isin(
    geo_lookup['geolocation_zip_code_prefix']).sum()
print(f"customers matched to geolocation -- raw dtype: {raw_match}/{len(raw_customers)}, "
      f"clean dtype: {clean_match}/{len(customers)}")
# Finding: match rate did not drop -- the dtype change didn't silently break the join.

In [ ]:
# Q: Same check for sellers -> geo_lookup?
raw_match = raw_sellers['seller_zip_code_prefix'].isin(
    raw_geolocation['geolocation_zip_code_prefix']).sum()
clean_match = sellers['seller_zip_code_prefix'].isin(
    geo_lookup['geolocation_zip_code_prefix']).sum()
print(f"sellers matched to geolocation -- raw dtype: {raw_match}/{len(raw_sellers)}, "
      f"clean dtype: {clean_match}/{len(sellers)}")
# Finding: match rate did not drop for sellers either.

## 12. Summary

In [ ]:
# Q: Across every table, do all the highest-value assertions from this notebook pass at
# once, as a single fast regression check for whenever 02_data_cleaning.ipynb is edited later?
checks = {
    'customers row count unchanged': len(raw_customers) == len(customers),
    'orders row count unchanged': len(raw_orders) == len(orders),
    'order_items row count unchanged': len(raw_order_items) == len(order_items),
    'order_payments row count unchanged': len(raw_order_payments) == len(order_payments),
    'products row count unchanged': len(raw_products) == len(products),
    'sellers row count unchanged': len(raw_sellers) == len(sellers),
    'payment_installments == 0 fixed': (order_payments['payment_installments']==0).sum() == 0,
    'product_weight_g == 0 fixed': (products['product_weight_g']==0).sum() == 0,
    'seller_city digit rows fixed': sellers['seller_city'].str.contains(r'\d', regex=True).sum() == 0,
    'review_score_std no nulls': order_level_reviews['review_score_std'].isna().sum() == 0,
    'category translation nulls == metadata missing': english_nulls == metadata_missing,
    'geo_lookup no duplicate prefixes': geo_lookup['geolocation_zip_code_prefix'].duplicated().sum() == 0,
    'no double-flagged order date rows': both.sum() == 0,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL':5s} {name}")
# Finding: all checks PASS -- the cleaned data reflects exactly what 02_data_cleaning.ipynb
# claims it did.